# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the FAIR\ :superscript:`2`\ dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library via its Croissant schema. 

### Dataset Source
The dataset is defined using the Croissant metadata standard and is available via the following URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

We will load the dataset metadata and available record sets with `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the Croissant Dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Review available record sets, fields, and their `@id`s.

**Note**: `mlcroissant` allows you to inspect the structure of the dataset, including all record sets and their fields. For reproducibility, we will list all record sets and for each, show the available fields (referenced by their `@id`).

In [ ]:
# Retrieve all record sets and display their @id and fields
record_set_ids = []

print("Record sets available in this dataset:")
for record_set in dataset.record_sets:
    print(f"- @id: {record_set['@id']} | name: {record_set.get('name', '[unnamed]')}")
    record_set_ids.append(record_set['@id'])
    
    if 'field' in record_set:
        print("  Fields:")
        for field in record_set['field']:
            if isinstance(field, dict):
                print(f"    - {field.get('@id', '[no_id]')}: {field.get('name', '[no_name]')}")
            else:
                print(f"    - {field}")
    print()

## 3. Data Extraction

Load data from available record sets into `pandas` DataFrames for further analysis. All access is performed by `@id` according to the Croissant schema.

If you're running this notebook for the first time, you may want to use just one record set initially; however, here we demonstrate iterating through all for generality.

In [ ]:
# Load data from all record sets into a dictionary of DataFrames
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            dataframes[record_set_id] = pd.DataFrame(records)
            print(f"Loaded DataFrame for record set: {record_set_id} (shape: {dataframes[record_set_id].shape})")
            print(f"  Columns: {dataframes[record_set_id].columns.tolist()}")
    except Exception as e:
        print(f"Warning: Could not load records for record set @id {record_set_id}: {e}")

if not dataframes:
    print("No record sets contain data or there are no record sets defined.")
else:
    # Display first 5 rows of the first available DataFrame
    first_id = list(dataframes.keys())[0]
    print(f"\nFirst DataFrame preview for record set {first_id}:")
    display(dataframes[first_id].head())

## 4. Exploratory Data Analysis (EDA)

Perform basic data processing steps using field `@id` values. You might select a numeric field for filtering and normalization, followed by group-based operations. For demonstration, we query the columns of the first data-containing record set and use its first numeric column.

In [ ]:
import numpy as np

if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]

    # Automatically select the first numeric field by attempting conversion
    numeric_field = None
    for col in df.columns:
        try:
            ser = pd.to_numeric(df[col], errors='coerce')
            # Check if sufficient non-NaN after conversion
            if ser.notnull().sum() > 0:
                numeric_field = col
                break
        except Exception:
            continue

    if numeric_field:
        print(f"Using numeric field (by @id): {numeric_field}")

        ser = pd.to_numeric(df[numeric_field], errors='coerce')
        threshold = ser.mean() if not np.isnan(ser.mean()) else 0
        filtered_df = df[ser > threshold].copy()
        filtered_df[numeric_field] = pd.to_numeric(filtered_df[numeric_field], errors='coerce')

        print(f"Filtered records where {numeric_field} > {threshold:.3f}:")
        display(filtered_df.head())

        normalized_col = f"{numeric_field}_normalized"
        filtered_df[normalized_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, normalized_col]].head())

        # Attempt to find a categorical/groupable field
        potential_group_fields = [c for c in df.columns if c != numeric_field]
        group_field = None
        for field in potential_group_fields:
            nunique = df[field].nunique(dropna=True)
            if nunique > 1 and nunique <= min(8, len(df)//4):  # heuristic
                group_field = field
                break

        if group_field:
            # Sanitize numeric columns for groupby
            for col in [numeric_field, normalized_col]:
                filtered_df[col] = pd.to_numeric(filtered_df[col], errors='coerce')
            grouped_df = filtered_df.groupby(group_field)[[numeric_field, normalized_col]].mean().reset_index()
            print(f"\nGrouped normalized {numeric_field} by categorical field (by @id): {group_field}")
            display(grouped_df.head())
        else:
            print("No suitable group field found in data for demonstration.")
    else:
        print("No numeric field found in the first available record set; cannot continue demo EDA.")
else:
    print("No loaded data available for EDA.")

## 5. Visualization

Visualize numeric field distribution and optionally group summary (if a grouping field was found).

In [ ]:
import matplotlib.pyplot as plt

if dataframes and 'filtered_df' in locals() and numeric_field:
    fig, ax = plt.subplots(figsize=(8, 4))
    filtered_df[numeric_field].hist(bins=20, ax=ax)
    ax.set_title(f"Distribution of {numeric_field} (filtered)")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    if 'grouped_df' in locals() and group_field:
        plt.figure(figsize=(8,4))
        plt.bar(grouped_df[group_field].astype(str), grouped_df[normalized_col])
        plt.ylabel(normalized_col)
        plt.xlabel(group_field)
        plt.title(f"Mean normalized {numeric_field} by {group_field}")
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()

## 6. Conclusion

- This notebook demonstrated dataset access and exploration using the Croissant schema and `mlcroissant`.
- All data elements were referenced and manipulated by their schema `@id` for transparency and reproducibility.
- Processed record sets were loaded into DataFrames and subjected to simple EDA workflows as illustrations.
- The FAIR\ :superscript:`2`\ dataset supports research in rangeland management practice adoption and enables policy-relevant, data-driven insights.